# PyDI Data Integration Workflow: Music

This notebook demonstrates how PyDI is used for end-to-end data integration. We'll work with music datasets to showcase the data integration pipeline from schema and entity matching to data fusion.

## Table of Contents
  - [Datasets](#datasets)
- [Part 1: Schema Matching and Value Normalization](#part-1-schema-matching-and-value-normalization)
  - [Step 1: Load Target Schema and Normalization Spec](#step-1-load-target-schema-and-normalization-spec)
  - [Step 2: Load Source Datasets](#step-2-load-source-datasets)
  - [Step 3: LLM-Based Schema Matching](#step-3-llm-based-schema-matching)
  - [Step 4: Schema Matching Evaluation](#step-4-evaluate-schema-matching-against-gold-mapping)
  - [Step 5: Translate and Normalize](#step-5-translate-and-normalize)
- [Part 2: Data Loading and Profiling](#part-2-data-loading-and-profiling)
- [Part 3: Entity Matching](#part-3-entity-matching)
  - [Step 1: Blocking](#step-1-blocking)
  - [Step 2: Blocking Evaluation](#step-2-evaluate-blocking-against-ground-truth)
  - [Step 3: Entity Matching with Comparators](#step-3-entity-matching-with-comparators)
  - [Step 4: Entity Matching Evaluation](#step-4-evaluate-matching-against-ground-truth)
- [Part 4: Data Fusion](#part-4-data-fusion)
  - [Step 1: Define Fusion Strategy](#step-1-define-fusion-strategy)
  - [Step 2: Run Fusion](#step-2-run-fusion)
  - [Step 3: Data Fusion Evaluation](#step-3-evaluate-data-fusion)

## Part 1: Schema Matching and Value Normalization

In [ ]:
from pathlib import Path
import time
start_time = time.time()
# Paths relative to this notebook
NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR = NOTEBOOK_DIR / "input"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd
import json
from PyDI.schemamatching import LLMBasedSchemaMatcher, SchemaMappingEvaluator, SchemaTranslator
from PyDI.normalization import load_normalization_spec
from langchain_openai import ChatOpenAI

## Step 1: Load Target Schema and Normalization Spec

In [ ]:
# Load the JSON Schema (used for both matching and normalization)
with open(INPUT_DIR / "schemamatching" / "target_schema.json") as f:
    target_schema = json.load(f)

# Load NormalizationSpec from the same schema
spec = load_normalization_spec(INPUT_DIR / "schemamatching" / "target_schema.json")

spec.set_column("tracks", output_type="list")
spec.set_column("release-date", output_type="string")

target_columns = list(spec.columns.keys())

# Create empty target DataFrame for schema matching
df_target = pd.DataFrame(columns=target_columns)
df_target.attrs["dataset_name"] = "target_schema"


## Step 2: Load Source Datasets

In [ ]:
from PyDI.io import load_xml, load_csv, load_json
mbrainz = load_csv(INPUT_DIR / "data" / "musicbrainz.csv")
mbrainz.attrs["dataset_name"] = "musicbrainz"
mbrainz.head()

In [ ]:
import re
# Clean column names from XML namespaces
def strip_all_ns(col):
    """
    Remove ALL {namespace} prefixes from a column name string.
    Example:
      '{ns}medium-list_{ns}medium_{ns}track' -> 'medium-list_medium_track'
    """
    if not isinstance(col, str):
        return col
    # remove every occurrence of {...}
    return re.sub(r"\{[^}]+\}", "", col)

mbrainz = mbrainz.rename(columns=strip_all_ns)
mbrainz.head()

In [ ]:
lastfm = load_csv(INPUT_DIR / "data" / "lastfm.csv")
lastfm.attrs["dataset_name"] = "lastfm"
lastfm.head()

In [ ]:
discogs = load_csv(INPUT_DIR / "data" / "discogs.csv")
discogs.attrs["dataset_name"] = "discogs"
discogs.head()

## Step 3: LLM-Based Schema Matching

In [ ]:
from dotenv import load_dotenv
load_dotenv()

# Initialize matcher with target schema for better context
matcher = LLMBasedSchemaMatcher(
    chat_model=ChatOpenAI(model="gpt-5.5"),
    num_rows=40,
    target_schema=target_schema,
)

# Match mbrainz dataset
mbrainz_mapping = matcher.match(mbrainz, df_target)

mbrainz_mapping

In [ ]:
lastfm_mapping = matcher.match(lastfm, df_target)
lastfm_mapping

In [ ]:
discogs_mapping = matcher.match(discogs, df_target)
discogs_mapping

## Step 4: Evaluate Schema Matching Against Gold Mapping


In [ ]:
# Load the manually curated schema-matching gold standard
with open(INPUT_DIR / "schemamatching" / "sm_mapping_gold.json") as f:
    schema_mapping_gold = json.load(f)

schema_mapping_gold_df = pd.DataFrame(schema_mapping_gold["mappings"])
schema_mapping_predictions = pd.concat(
    [mbrainz_mapping, lastfm_mapping, discogs_mapping],
    ignore_index=True,
)

schema_matching_metrics = SchemaMappingEvaluator.evaluate(
    schema_mapping_predictions,
    schema_mapping_gold_df,
    complete=True,
)

schema_matching_summary = pd.DataFrame([
    {
        **schema_matching_metrics,
        "predicted_mappings": len(schema_mapping_predictions),
        "gold_mappings": len(schema_mapping_gold_df),
    }
])

per_source_schema_matching = pd.DataFrame([
    {
        "source_dataset": source_dataset,
        **SchemaMappingEvaluator.evaluate(
            schema_mapping_predictions[
                schema_mapping_predictions["source_dataset"] == source_dataset
            ],
            schema_mapping_gold_df[
                schema_mapping_gold_df["source_dataset"] == source_dataset
            ],
            complete=True,
        ),
    }
    for source_dataset in sorted(schema_mapping_gold_df["source_dataset"].unique())
])

display(schema_matching_summary)
per_source_schema_matching


## Step 5: Translate and Normalize


In [ ]:
translator = SchemaTranslator()

# Translate + normalize each dataset with its own mapping

spec.set_column("release-country", country_format="name")
spec.set_column("release-date", output_type="string")

discogs_normalized = translator.translate(
    discogs,
    discogs_mapping,
    normalize=spec,
    on_failure="keep"
)

lastfm_normalized = translator.translate(
    lastfm, lastfm_mapping,
    normalize=spec, on_failure="keep"
)

mbrainz_normalized = translator.translate(
    mbrainz, mbrainz_mapping,
    normalize=spec, on_failure="keep"
)

In [ ]:
discogs_cols = [c for c in target_columns if c in discogs_normalized.columns]
lastfm_cols = [c for c in target_columns if c in lastfm_normalized.columns]
mbrainz_cols = [c for c in target_columns if c in mbrainz_normalized.columns]

In [ ]:
# Only keep target columns
mbrainz = mbrainz_normalized[mbrainz_cols].copy()
lastfm = lastfm_normalized[lastfm_cols].copy()
discogs = discogs_normalized[discogs_cols].copy()

import ast

def parse_track_list(value):
    if isinstance(value, list):
        items = value
    elif pd.isna(value):
        return []
    elif isinstance(value, str):
        text = value.strip()
        if not text:
            return []
        try:
            parsed = ast.literal_eval(text)
            items = parsed if isinstance(parsed, list) else [parsed]
        except (SyntaxError, ValueError):
            items = [part.strip() for part in text.split("|")]
    else:
        items = [value]

    cleaned = []
    seen = set()
    for item in items:
        if item is None or pd.isna(item):
            continue
        title = str(item).strip()
        if not title:
            continue
        key = re.sub(r"\s+", " ", title.casefold())
        if key in seen:
            continue
        seen.add(key)
        cleaned.append(title)
    return cleaned

for dataset in (mbrainz, discogs, lastfm):
    if "tracks" in dataset.columns:
        dataset["tracks"] = dataset["tracks"].apply(parse_track_list)


In [ ]:
# Keep release dates in the target schema format (YYYY-MM-DD) before translation.
def normalize_iso_date_column(df, column):
    if column not in df.columns:
        return df

    original = df[column]
    text = original.astype("string").str.strip().str.replace(r"-00", "-01", regex=True)
    text = text.str.replace(r"^([0-9]{4})$", r"\1-01-01", regex=True)
    text = text.str.replace(r"^([0-9]{4}-[0-9]{2})$", r"\1-01", regex=True)
    missing = original.isna() | text.eq("")
    parsed = pd.to_datetime(text, errors="coerce", format="%Y-%m-%d")
    normalized = parsed.dt.strftime("%Y-%m-%d")
    df[column] = normalized.where(parsed.notna(), original)
    df.loc[missing, column] = pd.NA
    return df


for dataset in (mbrainz, discogs, lastfm):
    normalize_iso_date_column(dataset, "release-date")

discogs.iloc[0]["release-date"]

In [ ]:
# --- MaDI-Bench normalization output: the normalized source tables, one CSV per source
# (record id + target-schema columns), the input of madi_bench.evaluation.score_normalization,
# which scores them against input/normalization/<split>.csv.
import json as _json
from pathlib import Path as _Path
_norm_dir = OUTPUT_DIR / "normalization"
_norm_dir.mkdir(parents=True, exist_ok=True)
_target = list(_json.loads((_Path(NOTEBOOK_DIR) / "input" / "schemamatching" / "target_schema.json").read_text())["properties"])
for _name, _df in {"musicbrainz": mbrainz, "discogs": discogs, "lastfm": lastfm}.items():
    _id_col = "id" if "id" in _df.columns else next((c for c in _df.columns if c.endswith("_id")), _df.columns[0])
    _cols = [_id_col] + [c for c in _df.columns if c in _target and c != "id" and c != _id_col]
    _df[_cols].rename(columns={_id_col: "id"}).to_csv(_norm_dir / f"{_name}.csv", index=False)
    print(f"normalized table {_name}: {len(_df)} rows, {len(_cols)} columns -> {_norm_dir / (_name + '.csv')}")


## Part 2: Data Loading and Profiling

In [ ]:
# Display basic information
datasets = [discogs, mbrainz, lastfm]
names = ["Discogs", "MusicBrainz", "Last.fm"]

total_records = sum(len(df) for df in datasets)
print(f"Total records across all datasets: {total_records:,}")

In [ ]:
from PyDI.utils import DataProfiler

# Initialize the DataProfiler
profiler = DataProfiler()

for df, name in zip(datasets, names):
    profile = profiler.summary(df) # automatically prints some statistics and returns object containing stats

display(profile)

## Part 3: Entity Matching

### Step 1: Blocking

In [ ]:
# Set up logging
import logging

import os
os.makedirs('output/logs', exist_ok=True)

logging.basicConfig(
    level=logging.WARNING, # Alternatively, use logging.DEBUG for more verbosity
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler('output/logs/pydi.log'),  # Save to file
          logging.StreamHandler()                      # Display on console
      ],
    force=True
)

In [ ]:
# Import blocking methods
from PyDI.entitymatching import StandardBlocker, SortedNeighbourhoodBlocker, TokenBlocker, EmbeddingBlocker
import re

# Standard Blocking - Longest Token in Name
# Add name_longest_token directly to the original dataframes
def get_longest_token(name):
    tokens = re.split(r"[^A-Za-z0-9_']+", str(name))    
    tokens = [t for t in tokens if t]
    return max(tokens, key=len) if tokens else ''

mbrainz['name_longest_token'] = mbrainz['name'].apply(get_longest_token)
discogs['name_longest_token'] = discogs['name'].apply(get_longest_token)
lastfm['name_longest_token'] = lastfm['name'].apply(get_longest_token)

standard_blocker_m2d = StandardBlocker(
    mbrainz, discogs,
    on=['name_longest_token'],
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
 )

standard_blocker_m2l = StandardBlocker(
    mbrainz, lastfm,
    on=['name_longest_token'],
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
 )

### Step 2: Evaluate Blocking Against Ground Truth

In [ ]:
import pandas as pd
from PyDI.io import load_csv
from PyDI.entitymatching import EntityMatchingEvaluator

# Evaluate the blocker against the held-out test pairs for each dataset pair
pairs = {
    "m2d": (standard_blocker_m2d, "musicbrainz_2_discogs_val.csv"),
    "m2l": (standard_blocker_m2l, "musicbrainz_2_lastfm_val.csv"),
}

blocking_results = {}
for pair_name, (blocker, gt_file) in pairs.items():
    test_gt = load_csv(
        INPUT_DIR / "entitymatching" / gt_file,
        name=f"test_{pair_name}", header=None, names=['id1', 'id2', 'label'], add_index=False,
    )
    blocking_results[pair_name] = EntityMatchingEvaluator.evaluate_blocking_batched(
        blocker=blocker,
        test_pairs=test_gt,
        out_dir=OUTPUT_DIR / "blocking-evaluation" / pair_name,
    )

blocking_summary = pd.DataFrame(
    [{"dataset_pair": name, **res} for name, res in blocking_results.items()]
)[["dataset_pair", "pair_completeness", "reduction_ratio"]]

print("Summary of Blocking Results:")
display(blocking_summary)

### Step 3: Entity Matching with Comparators

In [ ]:
from PyDI.entitymatching import StringComparator, DateComparator, NumericComparator

# ignore case and punctuation
def normalize_text(s: str) -> str: 
    if s is None:
        return ""
    return re.sub(r"[^\w\s]|_", "", s).lower()

comparators = [
    # Release name — Jaccard
    StringComparator(
        column='name', 
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    # Release artist — Jaccard
    StringComparator(
        column='artist',
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    # Release duration — within 10% --> allow 10% deviation
    NumericComparator(
        column='duration',
        method='relative_difference',
        max_difference=0.10
    ),
    # # Track list — overlap
    StringComparator(
        column='tracks',
        similarity_function='jaccard',
        preprocess=normalize_text,
        list_strategy="set_overlap"
    ),
    # Release date — within 2 years
    DateComparator(
        column='release-date',
        max_days_difference=365 * 2
    ),
    # Release country — Jaccard
    StringComparator(
        column='release-country',
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
]

In [ ]:
import numpy as np

# Convert lists in mbrainz["duration"] to single integer values (sum if list, else int)

def sum_duration(val):
    if isinstance(val, list):
        return int(np.nansum([int(x) for x in val if str(x).isdigit()]))
    try:
        return int(val)
    except Exception:
        return np.nan

mbrainz["duration"] = mbrainz["duration"].apply(sum_duration)

In [ ]:
from PyDI.entitymatching import RuleBasedMatcher

# Run rule-based matching on each candidate pool. lastfm lacks release-date and release-country,
# so it uses a reduced comparator list.
matcher = RuleBasedMatcher()
matching_pairs = {
    "m2d": (mbrainz, discogs, standard_blocker_m2d, comparators, 0.5),
    "m2l": (mbrainz, lastfm, standard_blocker_m2l, comparators[:-2], 0.3),
}

correspondences = {}
for pair_name, (df_left, df_right, blocker, comps, threshold) in matching_pairs.items():
    correspondences[pair_name] = matcher.match(
        df_left=df_left,
        df_right=df_right,
        candidates=blocker,
        comparators=comps,
        weights=None,
        threshold=threshold,
        id_column='id',
    )

correspondences_m2d = correspondences["m2d"]
correspondences_m2l = correspondences["m2l"]

### Step 4: Evaluate Matching Against Ground Truth

In [ ]:
from PyDI.entitymatching import MaximumBipartiteMatching

# Evaluate raw and 1:1-refined matching for each dataset pair
clusterer = MaximumBipartiteMatching()
gt_files = {
    "m2d": "musicbrainz_2_discogs_test.csv",
    "m2l": "musicbrainz_2_lastfm_test.csv",
}

matching_rows = []
correspondences_refined = {}
for pair_name, gt_file in gt_files.items():
    gt_test = load_csv(
        INPUT_DIR / "entitymatching" / gt_file,
        name=f"test_{pair_name}", header=None, names=['id1', 'id2', 'label'], add_index=False,
    )
    raw = correspondences[pair_name]
    refined = clusterer.cluster(raw)
    correspondences_refined[pair_name] = refined

    raw_eval = EntityMatchingEvaluator.evaluate_matching(
        correspondences=raw, test_pairs=gt_test,
        out_dir=OUTPUT_DIR / "debug_results_entity_matching" / f"{pair_name}_raw",
    )
    refined_eval = EntityMatchingEvaluator.evaluate_matching(
        correspondences=refined, test_pairs=gt_test,
        out_dir=OUTPUT_DIR / "debug_results_entity_matching" / f"{pair_name}_mbm",
    )
    matching_rows.append({"dataset_pair": pair_name, "stage": "raw", **raw_eval})
    matching_rows.append({"dataset_pair": pair_name, "stage": "mbm_refined", **refined_eval})

correspondences_m2d_post = correspondences_refined["m2d"]
correspondences_m2l_post = correspondences_refined["m2l"]

matching_summary = pd.DataFrame(matching_rows)[["dataset_pair", "stage", "precision", "recall", "f1", "accuracy"]]
print("Matching results (raw vs MBM-refined):")
display(matching_summary)

## Part 4: Data Fusion

In [ ]:
mbrainz["mbrainz_id"] = mbrainz["id"]

# Assign trust scores to datasets
mbrainz.attrs["trust_score"] = 3
discogs.attrs["trust_score"] = 1
lastfm.attrs["trust_score"] = 2

all_correspondences = pd.concat([correspondences_m2d, correspondences_m2l], ignore_index=True)
print(f'Total correspondences: {len(all_correspondences):,}')

## Step 1: Define Fusion Strategy 

In [ ]:
from PyDI.fusion import DataFusionStrategy, longest_string, shortest_string, prefer_higher_trust, maximum

strategy = DataFusionStrategy('music_fusion_strategy')

def prefer_track_list_by_source(values, *, sources=None, source_datasets=None, **kwargs):
    """Pick one coherent track list instead of unioning near-duplicate track titles."""
    source_priority = ["musicbrainz", "discogs", "lastfm"]
    candidates = []
    for value, record_id in zip(values, sources or []):
        tracks = parse_track_list(value)
        if not tracks:
            continue
        dataset = (source_datasets or {}).get(record_id, "unknown")
        priority = source_priority.index(dataset) if dataset in source_priority else len(source_priority)
        candidates.append((priority, -len(tracks), dataset, record_id, tracks))

    if not candidates:
        return None, 0.0, {"rule": "prefer_track_list_by_source", "reason": "no_valid_track_lists"}

    priority, _, dataset, record_id, tracks = sorted(candidates)[0]
    confidence = 1.0 if priority == 0 else max(0.5, 1.0 - 0.2 * priority)
    return tracks, confidence, {
        "rule": "prefer_track_list_by_source",
        "selected_dataset": dataset,
        "selected_record_id": record_id,
        "num_tracks": len(tracks),
    }

strategy.add_attribute_fuser('name', shortest_string)
strategy.add_attribute_fuser('artist', longest_string)
strategy.add_attribute_fuser('release-date', prefer_higher_trust)
strategy.add_attribute_fuser('release-country', prefer_higher_trust)
strategy.add_attribute_fuser('duration', maximum)
strategy.add_attribute_fuser('tracks', prefer_track_list_by_source)
strategy.add_attribute_fuser('label', longest_string)


## Step 2: Run Fusion

In [ ]:
from PyDI.fusion import DataFusionEngine

engine = DataFusionEngine(strategy, debug=True, debug_format='json',debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion.jsonl")

fused = engine.run(
    datasets=[mbrainz, discogs, lastfm],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False,
)

def musicbrainz_cluster_id(sources):
    if not isinstance(sources, (list, tuple, set)):
        return None
    for source_id in sources:
        source_id = str(source_id)
        if source_id.startswith("mbrainz_"):
            return source_id
    return None

# Align fused records with the gold fusion sets, which use the MusicBrainz id.
# Example: [mbrainz_1, discogs_3, lastFM_1] should evaluate as id == mbrainz_1.
fused["id"] = fused["_fusion_sources"].apply(musicbrainz_cluster_id)
fused = fused.dropna(subset=["id"]).copy()

print(f'Fused rows: {len(fused):,}')
display(fused.head(5))


In [ ]:
from PyDI.evaluation import evaluate_schema_consistency, write_schema_consistency_report

consistency_result = evaluate_schema_consistency(
    fused,
    INPUT_DIR / "schemamatching" / "target_schema.json",
    taxonomy_base_path=NOTEBOOK_DIR,
)
write_schema_consistency_report(
    consistency_result,
    OUTPUT_DIR / "metrics" / "consistency.json",
    metadata={
        "usecase": "music",
        "schema_path": str(INPUT_DIR / "schemamatching" / "target_schema.json"),
        "fusion_output_path": str(OUTPUT_DIR / "data_fusion" / "debug_fusion.jsonl"),
        "rows_evaluated": len(fused),
        "fused_table_source": "music_workflow.fused",
        "identifier_columns_excluded": True,
        "open_taxonomy_policy": "non_exhaustive_membership_is_diagnostic_not_penalizing",
    },
)
consistency_result["consistency_score"]

## Step 3: Evaluate Data Fusion

In [ ]:
from PyDI.fusion import tokenized_match, year_only_match, set_equality_match, numeric_tolerance_match, intersection


strategy.add_evaluation_function("name", tokenized_match)
strategy.add_evaluation_function("artist", tokenized_match)
strategy.add_evaluation_function("duration", numeric_tolerance_match, tolerance=10) # 10 seconds tolerance
strategy.add_evaluation_function("release-date", year_only_match)
strategy.add_evaluation_function("release-country", tokenized_match)
strategy.add_evaluation_function("label", tokenized_match)
strategy.add_evaluation_function("tracks", tokenized_match)

In [ ]:
# --- MaDI-Bench fusion evaluation: the public evaluation package scores the
# fused table on the validation and the test split -- strict comparators on the v2 gold set,
# all-gold accuracy (a gold record the pipeline did not produce counts as wrong) with its coverage.
# This is the scorer the benchmark's reference numbers were produced with; the notebook's own
# comparator registrations above are not used for scoring.
import json as _json, os as _os, sys as _sys
from pathlib import Path as _Path
try:
    from madi_bench.evaluation import score_fusion as _score_fusion
except ImportError:
    _sys.path.insert(0, _os.environ.get("MADI_BENCH_ROOT") or str(_Path(NOTEBOOK_DIR).resolve().parents[2]))
    from madi_bench.evaluation import score_fusion as _score_fusion
_fused_out = fused.copy()
# the notebook's own column names for target-schema attributes (the same aliases the benchmark's
# rescoring applies, reproduction/scoring/rescore.py): scoring is by target-schema name
for _src, _dst in ():
    if _src in _fused_out.columns and _dst not in _fused_out.columns:
        _fused_out = _fused_out.rename(columns={_src: _dst})
_fusion_dir = OUTPUT_DIR / "data_fusion"
_fusion_dir.mkdir(parents=True, exist_ok=True)
_fused_out.to_csv(_fusion_dir / "fused.csv", index=False)
_scores = {}
for _split in ("validation", "test"):
    _r = _score_fusion(NOTEBOOK_DIR, _fused_out, split=_split)
    _scores[_split] = _r
    print(f"{_split}: all-gold accuracy {_r['overall_accuracy_all_gold']:.4f} "
          f"(evaluated {_r['overall_accuracy_evaluated']:.4f} on {_r['n_gold_evaluated']}/{_r['n_gold']} gold records, "
          f"attribute cell coverage {_r['attribute_cell_coverage']:.2f})")
    for _a, _v in sorted(_r["per_attribute_accuracy_all_gold"].items()):
        print(f"    {_a:28s} {_v:.4f}")
(_fusion_dir / "fusion_scores_v2.json").write_text(_json.dumps(_scores, indent=1, default=str))


In [ ]:
end_time = time.time()
elapsed_time = end_time - start_time
print(f"\nTotal execution time: {elapsed_time:.2f} seconds")